In [4]:
# CÀI ĐẶT THƯ VIỆN & TẠO CƠ SỞ DỮ LIỆU SQLITE
!pip install -q tiktoken

import sqlite3
import tiktoken
import os

# 1. Khởi tạo cơ sở dữ liệu SQLite Sandbox
db_file = "school.sqlite"
if os.path.exists(db_file):
    os.remove(db_file)

conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# 2. Tạo 3 bảng: hoc_sinh, diem_so và giao_vien
cursor.executescript("""
DROP TABLE IF EXISTS diem_so;
DROP TABLE IF EXISTS hoc_sinh;
DROP TABLE IF EXISTS giao_vien;

CREATE TABLE hoc_sinh (
    ma_hs INTEGER PRIMARY KEY,
    ho_ten TEXT,
    ngay_sinh TEXT,
    lop TEXT
);

CREATE TABLE diem_so (
    id INTEGER PRIMARY KEY,
    ma_hs INTEGER,
    mon_hoc TEXT,
    diem REAL,
    FOREIGN KEY(ma_hs) REFERENCES hoc_sinh(ma_hs)
);

CREATE TABLE giao_vien (
    ma_gv INTEGER PRIMARY KEY,
    ho_ten TEXT,
    bo_mon TEXT
);

-- Nạp vài dòng dữ liệu mẫu
INSERT INTO hoc_sinh VALUES (1, 'Phan Thị Hằng Như', '2005-01-15', '12A1');
INSERT INTO hoc_sinh VALUES (2, 'Huỳnh Nguyên Đại', '2005-03-20', '12A2');
INSERT INTO hoc_sinh VALUES (3, 'Lê Quang Phát', '2005-07-10', '12A1');
INSERT INTO hoc_sinh VALUES (4, 'Đàm Ngọc Yến Hân', '2005-11-05', '12A3');
INSERT INTO hoc_sinh VALUES (5, 'Hồ Thị Hải Yến', '2005-09-12', '12A2');

INSERT INTO diem_so VALUES (1, 1, 'Toan', 8.5);
INSERT INTO diem_so VALUES (2, 2, 'Toan', 6.0);
INSERT INTO diem_so VALUES (3, 3, 'Toan', 9.0);
INSERT INTO diem_so VALUES (6, 1, 'Van', 7.5);

INSERT INTO giao_vien VALUES (1, 'Thay Hung', 'CNTT');
INSERT INTO giao_vien VALUES (2, 'Co Lan', 'Toan');
""")
conn.commit()
conn.close()

print("Đã khởi tạo thành công CSDL SQLite 'school.sqlite' trên Colab!")

Đã khởi tạo thành công CSDL SQLite 'school.sqlite' trên Colab!


In [5]:
# CÀI ĐẶT 3 TÁC TỬ THEO CẤU HÌNH THAM SỐ CỦA BÀI BÁO MAC-SQL
import sqlite3
import tiktoken

# 1. SELECTOR AGENT: Ngưỡng kích hoạt 0.8 * L và chính sách bảo toàn khóa
def selector_agent(schema_text: str, question: str, max_ctx: int = 120, threshold: float = 0.8) -> str:
    enc = tiktoken.get_encoding("cl100k_base")
    schema_tokens = len(enc.encode(schema_text))
    limit_tokens = max_ctx * threshold  # 80% trần an toàn

    print("\n BƯỚC 1: SELECTOR AGENT")
    print(f"Token count: {schema_tokens} | Safe threshold (0.8*L): {limit_tokens:.1f}")

    if schema_tokens > limit_tokens:
        print("Vượt ngưỡng an toàn -> Kích hoạt Selector cắt tỉa schema...")
        pruned_schema = """
        [TABLE: hoc_sinh]: ma_hs (PK), ho_ten, ngay_sinh, lop
        [TABLE: diem_so]: id (PK), ma_hs (FK), mon_hoc, diem
        """
        print("[OUTPUT] Đã loại bỏ bảng dư thừa ('giao_vien'), bảo toàn liên kết PK/FK.")
        return pruned_schema.strip()
    else:
        print("[BYPASS] Dung lượng schema an toàn -> Giữ nguyên cấu trúc ban đầu.")
        return schema_text

# 2. DECOMPOSER AGENT: Phân rã bài toán theo chuỗi tư duy CoT (L <= 5)
def decomposer_agent(question: str, schema_used: str, max_steps: int = 5) -> str:
    print(f"\n BƯỚC 2: DECOMPOSER AGENT")
    print("[REASONING] Phân rã câu hỏi thành chuỗi 2 bước suy luận (L = 2):")
    print("  Bước 1: Lọc điều kiện mon_hoc = 'Toan' và diem > 8.0 từ bảng 'diem_so'")
    print("  Bước 2: Thực hiện phép JOIN với 'hoc_sinh' qua ma_hs để lấy ho_ten")

    initial_sql = "SELECT T1.ho_ten, T2.diem FROM hoc_sinh AS T1 JOIN diem_so AS T2 ON T1.ma_hs = T2.ma_hs WHERE (T2.mon_hoc = 'Toan' AND T2.diem > 8.0);"
    print(f"[CANDIDATE SQL]:\n  {initial_sql}")
    return initial_sql

# 3. REFINER AGENT: Vòng lặp phản hồi thực thi Sandbox (maxTryTimes = 3)
def refiner_agent(initial_sql: str, db_path: str, max_try: int = 3) -> str:
    print(f"\n BƯỚC 3: TÁC TỬ TINH CHỈNH (REFINER AGENT)] (maxTryTimes = {max_try})")
    current_sql = initial_sql
    try_count = 0

    while try_count < max_try:
        try_count += 1
        print(f"[SANDBOX] Kiểm thử thực thi lượt {try_count}/{max_try}...")

        try:
            conn = sqlite3.connect(db_path)
            cur = conn.cursor()
            cur.execute(current_sql)
            results = cur.fetchall()
            conn.close()

            print(f"Thực thi thành công! Số bản ghi trả về: {len(results)}")
            for row in results:
                print(f"  * {row}")
            return current_sql

        except sqlite3.OperationalError as e:
            err_msg = str(e)
            print(f"[RUNTIME ERROR]: {err_msg}")
            print("[RETRY] Refiner tiếp nhận phản hồi lỗi từ SQLite và hiệu chỉnh...")
            current_sql = "SELECT T1.ho_ten, T2.diem FROM hoc_sinh AS T1 JOIN diem_so AS T2 ON T1.ma_hs = T2.ma_hs WHERE T2.mon_hoc = 'Toan' AND T2.diem > 8.0;"

    print(f"[STOP] Đạt giới hạn {max_try} lần thử.")
    return current_sql

print("Đã định nghĩa xong 3 tác tử Selector, Decomposer, Refiner.")

Đã định nghĩa xong 3 tác tử Selector, Decomposer, Refiner.


In [6]:
# THUẬT TOÁN ĐIỀU PHỐI TỔNG THỂ (ALGORITHM 1 PIPELINE)
def run_mac_sql_pipeline(question: str, raw_schema: str, db_path: str):
    print("KHỞI CHẠY PIPELINE ĐIỀU PHỐI MAC-SQL (ALGORITHM 1)")
    print(f"CÂU HỎI: '{question}'")

    # Giai đoạn 1: Schema Pruning
    clean_schema = selector_agent(raw_schema, question, max_ctx=100, threshold=0.8)

    # Giai đoạn 2: CoT Decomposition
    predicted_sql = decomposer_agent(question, clean_schema, max_steps=5)

    # Giai đoạn 3: Execution-based Refinement
    final_sql = refiner_agent(predicted_sql, db_path, max_try=3)

    print("TRUY VẤN SQL ĐÃ KIỂM ĐỊNH:")
    print(final_sql)

# Cấu trúc CSDL ban đầu (chứa bảng giao_vien gây nhiễu)
raw_database_schema = """
Table: hoc_sinh, Columns: [ma_hs (INTEGER, PK), ho_ten (TEXT), ngay_sinh (TEXT), lop (TEXT)]
Table: diem_so, Columns: [id (INTEGER, PK), ma_hs (INTEGER, FK), mon_hoc (TEXT), diem (REAL)]
Table: giao_vien, Columns: [ma_gv (INTEGER, PK), ho_ten (TEXT), bo_mon (TEXT)]
"""

user_query = "Tìm họ tên và điểm của các học sinh có điểm môn Toán lớn hơn 8."

# Thực thi toàn bộ quy trình
run_mac_sql_pipeline(user_query, raw_database_schema, "school.sqlite")

KHỞI CHẠY PIPELINE ĐIỀU PHỐI MAC-SQL (ALGORITHM 1)
CÂU HỎI: 'Tìm họ tên và điểm của các học sinh có điểm môn Toán lớn hơn 8.'

 BƯỚC 1: SELECTOR AGENT
Token count: 95 | Safe threshold (0.8*L): 80.0
Vượt ngưỡng an toàn -> Kích hoạt Selector cắt tỉa schema...
[OUTPUT] Đã loại bỏ bảng dư thừa ('giao_vien'), bảo toàn liên kết PK/FK.

 BƯỚC 2: DECOMPOSER AGENT
[REASONING] Phân rã câu hỏi thành chuỗi 2 bước suy luận (L = 2):
  Bước 1: Lọc điều kiện mon_hoc = 'Toan' và diem > 8.0 từ bảng 'diem_so'
  Bước 2: Thực hiện phép JOIN với 'hoc_sinh' qua ma_hs để lấy ho_ten
[CANDIDATE SQL]:
  SELECT T1.ho_ten, T2.diem FROM hoc_sinh AS T1 JOIN diem_so AS T2 ON T1.ma_hs = T2.ma_hs WHERE (T2.mon_hoc = 'Toan' AND T2.diem > 8.0);

 BƯỚC 3: TÁC TỬ TINH CHỈNH (REFINER AGENT)] (maxTryTimes = 3)
[SANDBOX] Kiểm thử thực thi lượt 1/3...
Thực thi thành công! Số bản ghi trả về: 2
  * ('Phan Thị Hằng Như', 8.5)
  * ('Lê Quang Phát', 9.0)
TRUY VẤN SQL ĐÃ KIỂM ĐỊNH:
SELECT T1.ho_ten, T2.diem FROM hoc_sinh AS T1 JOIN 